# Week 2 — Event-Level KSI Cleaning

Goal: transform the raw person-level Toronto KSI dataset into a reproducible
one-row-per-collision event table.

Grain: one row = one collision event.

Primary key: collision_id.

In [3]:
import pandas as pd

ksi_df = pd.read_csv("../data/raw/Motor Vehicle Collisions with KSI Data - 4326.csv")

ksi_df.head()

,_id,collision_id,accdate,stname1,stname2,stname3,per_inv,acclass,accloc,traffictl,...,distracted,cyclist,motorcyclist,other_micromobility,older_adult,pedestrian,red_light,school_child,heavy_truck,geometry
0,1,2006:893184,2006-01-01T02:36:00,WOODBINE AVE,O CONNOR DR,NaN,8,Non-Fatal Injury,Intersection-Related,No Control,...,False,False,False,False,False,False,False,True,False,"{""coordinates"": [[-79.31879768423914, 43.69959..."
1,2,2006:893184,2006-01-01T02:36:00,WOODBINE AVE,O CONNOR DR,NaN,8,Non-Fatal Injury,Intersection-Related,No Control,...,False,False,False,False,False,False,False,True,False,"{""coordinates"": [[-79.31879768423914, 43.69959..."
2,3,2006:893184,2006-01-01T02:36:00,WOODBINE AVE,O CONNOR DR,NaN,8,Non-Fatal Injury,Intersection-Related,No Control,...,False,False,False,False,False,False,False,True,False,"{""coordinates"": [[-79.31879768423914, 43.69959..."
3,4,2006:893184,2006-01-01T02:36:00,WOODBINE AVE,O CONNOR DR,NaN,8,Non-Fatal Injury,Intersection-Related,No Control,...,False,False,False,False,False,False,False,True,False,"{""coordinates"": [[-79.31879768423914, 43.69959..."
4,5,2006:893184,2006-01-01T02:36:00,WOODBINE AVE,O CONNOR DR,NaN,8,Non-Fatal Injury,Intersection-Related,No Control,...,False,False,False,False,False,False,False,True,False,"{""coordinates"": [[-79.31879768423914, 43.69959..."


In [4]:
ksi_df.shape

(20771, 50)

In [5]:
print("Число уникальных ДТП ksi:", ksi_df.collision_id.nunique())
print("Число повторов ksi:", ksi_df.collision_id.duplicated().sum())

Число уникальных ДТП ksi: 7620
Число повторов ksi: 13151


In [6]:
ksi_df.columns

Index(['_id', 'collision_id', 'accdate', 'stname1', 'stname2', 'stname3',
       'per_inv', 'acclass', 'accloc', 'traffictl', 'impactype', 'visible',
       'light', 'rdsfcond', 'road_class', 'failtorem', 'longitude', 'latitude',
       'veh_no', 'vehtype', 'initdir', 'per_no', 'invage', 'injury',
       'safequip', 'drivact', 'drivcond', 'pedact', 'pedcond', 'manoeuvre',
       'pedtype', 'cyclistype', 'cycact', 'cyccond', 'road_user', 'fatal_no',
       'wardname', 'division', 'neighbourhood', 'aggressive', 'distracted',
       'cyclist', 'motorcyclist', 'other_micromobility', 'older_adult',
       'pedestrian', 'red_light', 'school_child', 'heavy_truck', 'geometry'],
      dtype='str')

In [7]:
event_cols = [
    "accdate",
    "stname1",
    "stname2",
    "stname3",
    "acclass",
    "accloc",
    "traffictl",
    "impactype",
    "visible",
    "light",
    "rdsfcond",
    "road_class",
    "longitude",
    "latitude",
    "division",
    "neighbourhood",
]
consistency = ksi_df.groupby("collision_id")[event_cols].nunique(dropna=True)

consistency.head()

,accdate,stname1,stname2,stname3,acclass,accloc,traffictl,impactype,visible,light,rdsfcond,road_class,longitude,latitude,division,neighbourhood
collision_id,,,,,,,,,,,,,,,,
2006:882024,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,1
2006:882079,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1
2006:882174,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1
2006:882497,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1
2006:882501,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,1


In [8]:
consistency.shape

(7620, 16)

In [9]:
conflicts = (consistency > 1).sum()

conflicts

accdate          0
stname1          0
stname2          0
stname3          0
acclass          0
accloc           0
traffictl        0
impactype        0
visible          0
light            0
rdsfcond         0
road_class       0
longitude        0
latitude         0
division         0
neighbourhood    0
dtype: int64

In [10]:
event_df = ksi_df[["collision_id"] + event_cols].drop_duplicates(subset="collision_id")

event_df.shape

(7620, 17)

In [11]:
event_df.collision_id.nunique()

7620

In [12]:
event_df.duplicated().sum()

np.int64(0)

In [13]:
ksi_df.accdate = pd.to_datetime(ksi_df.accdate)
print("Временной промежуток в DF ksi:", ksi_df.accdate.min(), "-", ksi_df.accdate.max())
event_df.accdate = pd.to_datetime(event_df.accdate)
print("Временной промежуток в DF event:", event_df.accdate.min(), "-", event_df.accdate.max())

Временной промежуток в DF ksi: 2006-01-01 02:36:00 - 2026-09-15 13:16:00
Временной промежуток в DF event: 2006-01-01 02:36:00 - 2026-09-15 13:16:00


In [14]:
event_df.dtypes

collision_id                str
accdate          datetime64[us]
stname1                     str
stname2                     str
stname3                     str
acclass                     str
accloc                      str
traffictl                   str
impactype                   str
visible                     str
light                       str
rdsfcond                    str
road_class                  str
longitude               float64
latitude                float64
division                    str
neighbourhood               str
dtype: object

In [15]:
event_df.isna().sum()

collision_id        0
accdate             0
stname1             0
stname2           692
stname3          5635
acclass             1
accloc           1841
traffictl          75
impactype          11
visible            36
light              26
rdsfcond           38
road_class        327
longitude           1
latitude            1
division           25
neighbourhood      45
dtype: int64

In [16]:
(event_df.isna().mean() * 100).sort_values(ascending=False)

stname3          73.950131
accloc           24.160105
stname2           9.081365
road_class        4.291339
traffictl         0.984252
neighbourhood     0.590551
rdsfcond          0.498688
visible           0.472441
light             0.341207
division          0.328084
impactype         0.144357
latitude          0.013123
acclass           0.013123
longitude         0.013123
collision_id      0.000000
stname1           0.000000
accdate           0.000000
dtype: float64

In [17]:
event_df.to_csv("../data/interim/ksi_events.csv", index=False)

##  Data Dictionary

The following data dictionary documents the schema of the cleaned event-level
collision table.

For each field, it records:

- column name;
- meaning;
- data type;
- missingness;
- relevant notes or constraints.

The dictionary describes the dataset as it currently exists and does not define
future model features or the ML target.

| Column | Meaning | dtype | Missing | Missing % | Notes |
|---|---|---|---:|---:|---|
| `collision_id` | Unique identifier of a collision event | string | 0 | 0.00% | Primary key of the event-level table; one row per collision |
| `accdate` | Date and time when the collision occurred | datetime64 | 0 | 0.00% | Coverage: 2006-01-01 to 2026-09-15 |

In [20]:
data_dictionary_stats = pd.DataFrame({
    "Column": event_df.columns,
    "dtype": event_df.dtypes.astype(str).values,
    "Missing": event_df.isna().sum().values,
    "Missing %": (event_df.isna().mean().values * 100).round(2),
})

data_dictionary_stats

,Column,dtype,Missing,Missing %
0,collision_id,str,0,0.00
1,accdate,datetime64[us],0,0.00
2,stname1,str,0,0.00
3,stname2,str,692,9.08
4,stname3,str,5635,73.95
5,acclass,str,1,0.01
6,accloc,str,1841,24.16
7,traffictl,str,75,0.98
8,impactype,str,11,0.14
9,visible,str,36,0.47


In [21]:
from road_risk.data.events import build_event_table

pipeline_event_df = build_event_table(ksi_df)

pipeline_event_df.shape

(7620, 17)

In [22]:
pipeline_event_df["collision_id"].nunique()

7620

In [23]:
pipeline_event_df.equals(event_df)

True

## Conclusion

The Week 2 event-level cleaning pipeline was successfully validated.

Key results:

- the raw person-level KSI dataset was transformed into 7,620 collision-level events;
- `collision_id` is unique and defines the event-level table grain;
- selected event-level attributes contain no conflicting non-missing values within a collision;
- no exact duplicate event rows were found;
- `accdate` was converted to a datetime representation with no missing timestamps;
- spatial coordinates are available for 7,619 of 7,620 collision events;
- missingness was documented without arbitrary imputation;
- SQL validation independently confirmed the main dataset invariants;
- the reusable `build_event_table()` implementation reproduces the exploratory notebook result.

The resulting table is an intermediate event-level dataset and is not yet the
final machine-learning observation table.

No ML target or feature set is defined at this stage.